# 03d. Повторная валидация

Параметры заданы в коде. Запускайте из корня репозитория с `YANDEX_TASKS_OAUTH_TOKEN` в локальном `.env`. Зависимости: `python -m pip install -r requirements-tasks.txt`.

Ноутбук создаёт пул и загружает задания из локального файла. Пул открывается отдельно после подготовки заданий.

Сначала выполните `03a_create_annotation.ipynb` и `03b_create_validation.ipynb`. Используются их проекты и навыки. История контрольных ответов сохраняется общей с основной валидацией.

In [ ]:
import json
import os
from datetime import datetime, timedelta, timezone
from pathlib import Path
from uuid import uuid4

import requests
from dotenv import load_dotenv

load_dotenv()
api = requests.Session()
api.headers['Authorization'] = f"OAuth {os.environ['YANDEX_TASKS_OAUTH_TOKEN']}"
BASE_URL = 'https://tasks.yandex.ru/api/v1'


def create(path, payload, params=None):
    response = api.post(f'{BASE_URL}/{path}', json=payload, params=params, timeout=60)
    response.raise_for_status()
    return response.json()


In [ ]:
annotation_ids = json.loads(Path('annotation_ids.json').read_text(encoding='utf-8'))
validation_ids = json.loads(Path('validation_ids.json').read_text(encoding='utf-8'))

project_id = validation_ids['project_id']
participation_skill_id = annotation_ids['participation_skill_id']
exam_skill_id = validation_ids['exam_skill_id']
quality_skill_id = validation_ids['quality_skill_id']
control_history_id = validation_ids['control_history_id']

## Валидация переразметки 43 фотографий

Параметры: 1 ₽ за страницу, перекрытие 3, лимит 20 минут.

In [ ]:
correction_pool_payload = {'project_id': project_id,
 'private_name': 'BBox — ревью переразметки 43 фото v1',
 'public_description': 'Проверьте рамки вокруг взрослых бабочек. До 10 фотографий на странице.',
 'may_contain_adult_content': False,
 'reward_per_assignment': 1.0,
 'assignment_max_duration_seconds': 1200,
 'auto_accept_solutions': True,
 'auto_accept_period_day': 21,
 'assignments_issuing_config': {'issue_task_suites_in_creation_order': False},
 'filter': {'and': [{'category': 'computed',
                     'key': 'client_type',
                     'operator': 'EQ',
                     'value': 'BROWSER'},
                    {'category': 'profile', 'key': 'languages', 'operator': 'IN', 'value': 'RU'},
                    {'category': 'skill', 'key': participation_skill_id, 'operator': 'EQ'},
                    {'category': 'skill', 'key': exam_skill_id, 'operator': 'GTE', 'value': 80},
                    {'category': 'skill', 'key': quality_skill_id, 'operator': 'GTE', 'value': 80}]},
 'quality_control': {'configs': [{'collector_config': {'type': 'GOLDEN_SET',
                                                       'uuid': control_history_id,
                                                       'parameters': {'history_size': 10}},
                                  'rules': [{'conditions': [{'key': 'golden_set_answers_count',
                                                             'operator': 'GTE',
                                                             'value': 10}],
                                             'action': {'type': 'SET_SKILL_FROM_OUTPUT_FIELD',
                                                        'parameters': {'from_field': 'golden_set_correct_answers_rate',
                                                                       'skill_id': quality_skill_id}}}]}]},
 'defaults': {'default_overlap_for_new_task_suites': 3},
 'will_expire': (datetime.now(timezone.utc) + timedelta(days=30)).strftime('%Y-%m-%dT%H:%M:%SZ')}

In [ ]:
correction_pool = create('pools', correction_pool_payload)
correction_pool_id = correction_pool['id']
correction_pool_id

### Загрузка заданий

43 ответа переразметки и 5 контрольных заданий, 5 страниц. Перекрытие 3. Поле `source_assignment_id` связывает проверку с ответом разметчика.

In [ ]:
pages = json.loads(Path('tasks/revalidation.json').read_text(encoding='utf-8'))
for page in pages:
    create('task-suites', {'pool_id': correction_pool_id, **page}, params={'open_pool': 'false'})

print(f'Загружено страниц: {len(pages)}; заданий: {sum(len(p["tasks"]) for p in pages)}')

## Созданный ID

In [ ]:
revalidation_ids = {
    'project_id': project_id,
    'pool_id': correction_pool_id,
}
Path('revalidation_ids.json').write_text(
    json.dumps(revalidation_ids, ensure_ascii=False, indent=2), encoding='utf-8'
)
revalidation_ids